# 03 Manual Golden-set Answer Evaluation

Scope: this notebook uses 8 manually curated labor-law questions as a smoke/regression set. The scores are useful for quick comparison, not for a statistically robust benchmark.
For a broader benchmark, generate 30-50 synthetic questions with `ragas.testset.TestsetGenerator`, then manually review and freeze the accepted questions before scoring.

Notebook này chạy sau khi notebook 02 đã chọn `hybrid_rrf`.
Khi có `GEMINI_API_KEY` hoặc `GEMINI_API_KEYS`, notebook tự sinh answer mới bằng `gemini-3.1-flash-lite`, chấm `faithfulness`, `answer_relevancy`, `context_precision`, rồi ghi report.
Nếu chưa có key, notebook tải artifact đã commit để vẫn chạy được phần phân tích.

In [ ]:
import json
import os
import re
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from dotenv import load_dotenv

from src.data.chunking import chunk_documents
from src.data.load_dataset import load_documents_from_jsonl
from src.generation.gemini_client import BatchGeminiClient
from src.generation.prompt import build_rag_prompt
from src.retrieval.pipeline import RetrievalPipeline

load_dotenv(PROJECT_ROOT / ".env")
MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite")
HAS_GEMINI_KEY = bool(os.getenv("GEMINI_API_KEYS") or os.getenv("GEMINI_API_KEY"))
USE_TFIDF_FALLBACK = os.getenv("USE_TFIDF_FALLBACK", "false").lower() == "true"
CORPUS_PATH = PROJECT_ROOT / "data/processed/labor_corpus.jsonl"
ANSWERS_PATH = PROJECT_ROOT / "reports/rag_answers.json"
EVAL_JSON = PROJECT_ROOT / "reports/ragas_evaluation.json"
EVAL_MD = PROJECT_ROOT / "reports/ragas_evaluation.md"

In [ ]:
MANUAL_GOLDEN_QUESTIONS = [
    "Người lao động đơn phương chấm dứt hợp đồng lao động cần báo trước bao lâu?",
    "Trường hợp nào người lao động được nhận trợ cấp thôi việc?",
    "Doanh nghiệp có trách nhiệm gì về an toàn vệ sinh lao động?",
    "Quy định về tiền lương và lương tối thiểu của người lao động là gì?",
    "Người lao động nước ngoài cần điều kiện gì để làm việc tại Việt Nam?",
    "Kỷ luật sa thải người lao động được áp dụng trong trường hợp nào?",
    "Bảo hiểm thất nghiệp hỗ trợ người lao động như thế nào?",
    "Tranh chấp lao động tập thể được giải quyết như thế nào?",
]

print(f"Manual golden-set questions: {len(MANUAL_GOLDEN_QUESTIONS)}")

In [ ]:
if HAS_GEMINI_KEY:
    documents = load_documents_from_jsonl(str(CORPUS_PATH))
    retriever = RetrievalPipeline(chunk_documents(documents), use_tfidf_fallback=USE_TFIDF_FALLBACK)
    client = BatchGeminiClient(model=MODEL)
    answers = []
    for question in MANUAL_GOLDEN_QUESTIONS:
        results = retriever.retrieve(question, top_k=6)
        prompt, citations = build_rag_prompt(question, results)
        answers.append({"question": question, "answer": client.generate(prompt), "contexts": [result.chunk.text for result in results], "citations": citations})
    ANSWERS_PATH.write_text(json.dumps(answers, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Generated {len(answers)} answers with {MODEL}")
else:
    answers = json.loads(ANSWERS_PATH.read_text(encoding="utf-8"))
    print("No Gemini key configured: loaded committed answers artifact.")

pd.DataFrame([{"question": row["question"], "answer_preview": row["answer"][:160], "contexts": len(row["contexts"])} for row in answers])

In [ ]:
EVALUATOR_PROMPT = """Bạn là evaluator RAG tiếng Việt. Chấm điểm answer dựa trên question và contexts.
Trả lời CHỈ bằng JSON hợp lệ với các field: faithfulness, answer_relevancy, context_precision, reason.
Ba score là số từ 0.0 đến 1.0.

QUESTION:
{question}

ANSWER:
{answer}

CONTEXTS:
{contexts}
"""


def extract_json(text: str) -> dict:
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object in evaluator response: {text[:200]}")
    return json.loads(match.group(0))


def evaluate_answers(samples: list[dict]) -> dict:
    evaluator = BatchGeminiClient(model=MODEL)
    rows = []
    for sample in samples:
        contexts = "\n\n".join(f"[{idx + 1}] {text[:1800]}" for idx, text in enumerate(sample["contexts"][:6]))
        parsed = extract_json(evaluator.generate(EVALUATOR_PROMPT.format(question=sample["question"], answer=sample["answer"], contexts=contexts)))
        rows.append({"question": sample["question"], "faithfulness": float(parsed["faithfulness"]), "answer_relevancy": float(parsed["answer_relevancy"]), "context_precision": float(parsed["context_precision"]), "reason": parsed.get("reason", "")})
    aggregate = {metric: sum(row[metric] for row in rows) / len(rows) for metric in ["faithfulness", "answer_relevancy", "context_precision"]}
    aggregate.update({
        "samples": len(rows),
        "mode": "manual_golden_set_ragas_style_ai_studio_genai",
        "model": MODEL,
        "question_source": "manual_golden_set",
        "evaluation_scope": "smoke_regression_only",
        "benchmark_note": "Scores from 8 manually curated questions are useful for regression checks but are not a statistically robust benchmark.",
    })
    return {"aggregate": aggregate, "details": rows}

In [ ]:
if HAS_GEMINI_KEY:
    evaluation = evaluate_answers(answers)
    EVAL_JSON.write_text(json.dumps(evaluation, ensure_ascii=False, indent=2), encoding="utf-8")
    aggregate = evaluation["aggregate"]
    lines = [
        "# Manual Golden-set RAGAS-style Smoke Evaluation",
        "",
        f"Evaluator model: `{MODEL}` via AI Studio `google-genai`.",
        f"Question set: {aggregate['samples']} manually curated Vietnamese labor-law questions.",
        "Scope: smoke/regression check only; these scores are not a statistically robust benchmark.",
        "Next benchmark step: generate 30-50 synthetic questions with `ragas.testset.TestsetGenerator`, manually review them, freeze the accepted set, then re-run answer evaluation.",
        "",
        "| Metric | Score |",
        "|---|---:|",
        f"| Faithfulness | {aggregate['faithfulness']:.3f} |",
        f"| Answer relevancy | {aggregate['answer_relevancy']:.3f} |",
        f"| Context precision | {aggregate['context_precision']:.3f} |",
        "",
        "## Per-sample notes",
        "",
    ]
    for row in evaluation["details"]:
        lines.append(f"- **{row['question']}** - F={row['faithfulness']:.2f}, R={row['answer_relevancy']:.2f}, CP={row['context_precision']:.2f}. {row['reason']}")
    EVAL_MD.write_text("\n".join(lines), encoding="utf-8")
    print(f"Evaluated {len(answers)} answers with {MODEL}")
else:
    evaluation = json.loads(EVAL_JSON.read_text(encoding="utf-8"))
    print("No Gemini key configured: loaded committed evaluation artifact.")

pd.DataFrame([evaluation["aggregate"]])

In [ ]:
pd.DataFrame(evaluation["details"])[["question", "faithfulness", "answer_relevancy", "context_precision", "reason"]]

## Optional: synthetic RAGAS testset

The manual set above is intentionally small. For a stronger benchmark, create 30-50 synthetic questions with `ragas.testset.TestsetGenerator`, review them manually, and save only accepted questions as a frozen evaluation set.

Keep this optional path outside the core app requirements; install `ragas` only when generating the benchmark testset.

In [ ]:
RUN_SYNTHETIC_TESTSET = False
SYNTHETIC_TESTSET_SIZE = 40

if RUN_SYNTHETIC_TESTSET:
    from ragas.testset import TestsetGenerator

    documents = load_documents_from_jsonl(str(CORPUS_PATH))
    chunks = chunk_documents(documents)
    # Current RAGAS stable flow: build LangChain documents, then call
    # TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
    # and generator.generate_with_langchain_docs(docs, testset_size=SYNTHETIC_TESTSET_SIZE).
    print(f"Prepare {len(chunks)} chunks for TestsetGenerator and generate {SYNTHETIC_TESTSET_SIZE} reviewed questions.")
    raise NotImplementedError(
        "Install ragas, wire the generator for the current RAGAS version, and save only manually reviewed questions."
    )
else:
    print("Synthetic RAGAS testset generation disabled. Use 30-50 reviewed synthetic questions for the benchmark run.")

## Selected configuration for deploy

- Retrieval: **Hybrid RRF** production pipeline.
- Vector backend: **FAISS** with multilingual sentence embeddings; TF-IDF is only an offline smoke-test fallback.
- LLM: **`gemini-3.1-flash-lite`** via AI Studio `google-genai`.
- UI: Streamlit retrieval-only mode works without a key; answer generation requires a Gemini key.